# CHN Game Data Scraper: new 2025 version - incremental
- 

In [1]:
# ==========================================
# 0. Setup / Config
# ==========================================
import os
import time
import logging
from datetime import datetime, timedelta

import requests
from bs4 import BeautifulSoup
import pandas as pd
from tqdm import tqdm
from sqlalchemy import create_engine, inspect, text

import re
import numpy as np
from typing import Optional

# ----------------------------------------
## IMPORT dictionary, ect from config file
import config

recent_clean_db = config.recent_clean_db ## Red DB filepath from config.py
### Point to empty DB for testing to force full scrape
# recent_clean_db = "../data/db/EMPTY_TEST_DB.db"
# ----------------------------------------``


# --------- CONFIG ---------
SEASON = "2025-2026"
SCHEDULE_URL = "https://www.collegehockeynews.com/schedules/?season=20252026"
BASE_URL = "https://www.collegehockeynews.com"
current_year_url = SCHEDULE_URL

# Path to the *existing* cleaned DB you want to update
# recent_clean_db =  ## Currently imported from config.py

# Number of days back where we *allow* re-scrapes of already-seen games
# Set to None if you want to NEVER rescrape existing games.
RESCRAPE_WINDOW_DAYS = 3

# Canonical table we use to know what games are already in the DB
CANONICAL_GAME_TABLE = "game_details"

# All tables that store rows keyed by Game_ID for this pipeline
GAME_TABLES = [
    "game_details",
    "scoring_summary",
    "penalty_summary",
    "goalie_stats",
    "player_stats",
    "line_chart",
    "linescore",
]

# # Clear existing handlers (prevents double/triple logs)
# for handler in logging.root.handlers[:]:
#     logging.root.removeHandler(handler)

# logging.basicConfig(
#     filename=LOG_FILE,
#     level=logging.INFO,
#     format="%(asctime)s [%(levelname)s] %(message)s"
# )

# # Optional: also print to notebook output once (not 3×)
# console = logging.StreamHandler()
# console.setLevel(logging.INFO)
# logging.getLogger().addHandler(console)


In [2]:
# ### Check text content of config.py below ###
# # ==========================================
# # Print config.py content
# with open('config.py', 'r') as file:
#     config_content = file.read()
# print("Content of config.py:")
# print(config_content)

In [3]:
### Check path and connection to recent_clean_db below ###
# ==========================================
# Check if the recent_clean_db path exists
if not os.path.exists(config.recent_clean_db):
    logging.error(f"The database file does not exist at the specified path: {config.recent_clean_db}")

# Print relative and absolute paths
logging.info(f"Relative path to the database: {config.recent_clean_db}")
logging.info(f"Absolute path to the database: {os.path.abspath(config.recent_clean_db)}")

# Create a database engine
engine = create_engine(f"sqlite:///{config.recent_clean_db}")
# Test the database connection
try:
    with engine.connect() as connection:
        connection.execute(text("SELECT 1"))
    logging.info("Successfully connected to the database.")
except Exception as e:
    logging.error(f"Failed to connect to the database: {e}")
# ==========================================


## Helper Functions

### Cleaning and normalizing

In [4]:
def clean_name_columns(df: pd.DataFrame, columns_to_clean: list[str]) -> pd.DataFrame:
    """
    Clean up troublesome characters in specific columns:
    - '-' -> ' '
    - '.' removed
    - "'" removed
    - strip whitespace
    """
    for col in columns_to_clean:
        if col in df.columns:
            s = df[col].astype(str)
            s = s.str.replace('-', ' ', regex=False)
            s = s.str.replace('.', '', regex=False)
            s = s.str.replace("'", '', regex=False)
            s = s.str.strip()
            df[col] = s
    return df


TEAM_CLEAN_COLUMNS = {
    "advanced_metrics": ["Team"],
    "game_details": ["Away_Team", "Home_Team"],
    "goalie_stats": ["Team"],
    "line_chart": ["Team"],
    "linescore": ["Team"],
    "penalty_summary": ["Team"],
    "player_stats": ["Team"],
    "scoring_summary": ["Team"],
    # You can add 'Player' or others later if you decide to clean those too
    # e.g. "player_stats": ["Team", "Player"]
}


In [5]:
######### LEGACY FUNCTIONS BELOW #########

#### NEED TO POINT TO SCHOOL INFO FILE TO CREATE ABBREVIATION DICTIONARY ####
# Load school info DataFrame
## Load school infomation from arena_school_info.csv
school_info_df = pd.read_csv(os.path.join('..', 'data', 'school_info', 'arena_school_info.csv'))

# Create a dictionary for abbreviations to full team names
abbreviation_to_fullname = school_info_df.set_index('abv')['School'].to_dict()

# ==========================================
# create a function to replace abbreviations with full names
def replace_abbreviations_with_fullnames(df, column_name, abv_to_fullname_dict):
    """
    Replace team abbreviations in a specified column of a DataFrame with full team names.

    Parameters:
    df (pd.DataFrame): The DataFrame containing the column to be modified.
    column_name (str): The name of the column where abbreviations need to be replaced.
    abv_to_fullname_dict (dict): A dictionary mapping abbreviations to full team names.

    Returns:
    pd.DataFrame: The modified DataFrame with full team names.
    """
    df[column_name] = df[column_name].map(abv_to_fullname_dict).fillna(df[column_name])
    return df

In [6]:
# --- Column name canonicalization (per table) ---
PLAYER_STATS_COLMAP = {
    'Pt.': 'Pts',
    '+/-': 'PlusMinus',
    'Sh': 'Shots',
    'TOI': 'TOI',
    'PIM': 'PIM',
    'FOW': 'FOW',
    'FOL': 'FOL',
    'FO%': 'FO_Pct',
}

ADV_METRICS_COLMAP = {
    # period totals & per-period abbreviations
    'Bl': 'Blocks',
    'Mi': 'Misses',
    'SV': 'SOG',      # “shots on goal” / “saves” depending on site; keep consistent with your use
    'G':  'Goals',
    'TSA':'TotalSA',
    'BLKs':'BlocksTotal',
    'FO': 'Faceoffs',
}

SAFE_NAME = lambda s: re.sub(r'[^0-9a-zA-Z_]+', '_', s).strip('_')  # last resort cleaner

def normalize_columns(df: pd.DataFrame, table: str) -> pd.DataFrame:
    df = df.copy()
    # Strip spaces in headers
    df.columns = [c.strip() for c in df.columns]
    # Table-specific renames first
    if table == 'player_stats':
        df.rename(columns=PLAYER_STATS_COLMAP, inplace=True)
    elif table == 'advanced_metrics':
        df.rename(columns=ADV_METRICS_COLMAP, inplace=True)
    # Fallback: make every col SQLite-safe (no punctuation)
    df.rename(columns={c: SAFE_NAME(c) for c in df.columns}, inplace=True)
    return df

def drop_header_and_noise_rows(df: pd.DataFrame, table: str) -> pd.DataFrame:
    df = df.copy()
    # Common header-row pattern: “G”/“A” columns literally have “G”/“A”
    for gcol in ['G', 'Goals']:
        if gcol in df.columns and 'A' in df.columns:
            df = df[~((df[gcol].astype(str) == 'G') & (df['A'].astype(str) == 'A'))]
    # Drop rows where every non-ID stat equals the column labels or is empty
    if 'Player' in df.columns and 'Team' in df.columns:
        # A frequent artifact: team name in 'Player' for the header row
        df = df[~(df['Player'].astype(str).str.strip().eq(df['Team'].astype(str).str.strip())
                  & df.filter(regex=r'^(Pts|PlusMinus|Shots|TOI|PIM)$').isin(['Pts','+/-','Sh','TOI','PIM']).any(axis=1).fillna(False))]
    # Normalize empties
    df.replace({'': None, '—': None, '–': None, '-': '-', 'N/A': None}, inplace=True)
    return df


def ensure_table_columns(engine, table: str, columns: list[str]) -> None:
    insp = inspect(engine)
    if not insp.has_table(table):
        return
    with engine.begin() as conn:
        # rows like: (cid, name, type, notnull, dflt_value, pk)
        rows = conn.exec_driver_sql(f'PRAGMA table_info("{table}")').fetchall()
        existing_cols = {r[1] for r in rows}  # index 1 = column name
        for col in columns:
            if col not in existing_cols:
                conn.exec_driver_sql(f'ALTER TABLE "{table}" ADD COLUMN "{col}" TEXT')


def rename_duplicate_columns(df):
    df = df.copy()
    counts = {}
    new_cols = []
    for c in map(str, df.columns):  # ensure all are strings
        if c in counts:
            counts[c] += 1
            new_cols.append(f"{c}_{counts[c]}")
        else:
            counts[c] = 0
            new_cols.append(c)
    df.columns = new_cols
    return df


### Parse Yearly Schedule

In [7]:
# ## Functions
# ### Parse the current season schedule / results page

# def parse_current_season(url):
#     # Initialize variables
#     current_date = None
#     current_conference = None
#     game_notes = None

#     # Initialize an empty list to hold the data
#     data = []

#     # Parse the page with BeautifulSoup
#     # Get the page with requests
#     response = requests.get(url)

#     # Create a BeautifulSoup object
#     soup = BeautifulSoup(response.text, 'html.parser')

#     # select the table or tables
#     tables = soup.find_all('table')

#     rows = soup.find_all('tr')

#     # Loop through each row to find relevant information
#     for row in rows:
#         # Check for date row
#         if row.get('class') == ['stats-section']:
#             current_date = row.find('td').text.strip()
#         # Check for conference row
#         elif row.get('class') == ['sked-header']:
#             current_conference = row.find('td').text.strip()
#         # Check for game notes
#         elif len(row.find_all('td')) == 2:
#             game_notes = row.find_all('td')[1].text.strip()
#         # Process rows with game data
#         elif row.get('valign') == 'top':
#             cells = row.find_all('td')
#             if len(cells) >= 9:
#                 home_team = cells[0].text.strip()
#                 # Remove any hyphens from the team name
#                 home_team = home_team.replace('-', ' ')
#                 home_team_link = cells[0].find('a')['href'] if cells[0].find('a') else None
#                 home_score = cells[1].text.strip()
#                 away_team = cells[3].text.strip()
#                 away_team_link = cells[3].find('a')['href'] if cells[3].find('a') else None
#                 away_score = cells[4].text.strip()
#                 ot = cells[5].text.strip()
#                 box_link = cells[7].find('a')['href'] if cells[7].find('a') else None
#                 metrics_link = cells[8].find('a')['href'] if cells[8].find('a') else None
#                 # Capture Game Notes
#                 game_notes_cell = cells[-1].find('small')
#                 game_notes = game_notes_cell.text.strip() if game_notes_cell else None

#                 # Append data to the list
#                 data.append([current_date, current_conference, game_notes, home_team, home_team_link, home_score, away_team, away_team_link, away_score, ot, box_link, metrics_link])
#                 game_notes = None  # Reset game notes for the next row

#     return data

#     ## Turn the list-data into a dataframe
# ## call the function
# data = parse_current_season(current_year_url)


# # Create a dataframe from the list

# columns = ['Date', 'Conference', 'Game_Notes', 'Home_Team', 'Home_Team_Link', 'Home_Score', 'Away_Team', 'Away_Team_Link', 'Away_Score', 'OT', 'Box_Link', 'Metrics_Link']
# df = pd.DataFrame(data, columns=columns)
            
# ## Extract the day of the week from the date and save in new column
# df['Day'] = pd.to_datetime(df['Date']).dt.day_name()
# # remove day of the week from date
# # format data column as YYYY-MM-DD
# df['Date'] = pd.to_datetime(df['Date']).dt.strftime('%Y-%m-%d')

# ### Create a new column for the game ID
# ## Game ID will be a combination of the date and abbreviated team names

# # Loop to abbreviate the team names
# for row in df.itertuples():
#     home_team = row.Home_Team
#     away_team = row.Away_Team
#     home_team_abbr = home_team.split(' ')[-1]
#     away_team_abbr = away_team.split(' ')[-1]
#     # Remove any hyphens from the team name if there are any
#     home_team_abbr = home_team_abbr.replace('-', ' ')
#     away_team_abbr = away_team_abbr.replace('-', ' ')
#     game_id = f'{row.Date}-{home_team_abbr}-{away_team_abbr}'
#     df.loc[row.Index, 'Game_ID'] = game_id

# # Create a new column for the game ID
# df['Game_ID'] = df['Game_ID'].str.replace(',', '')

# # Remove any hyphens from the team names if any
# df['Home_Team'] = df['Home_Team'].str.replace('-', ' ')
# df['Away_Team'] = df['Away_Team'].str.replace('-', ' ')

# # Apply the function to the DataFrame
# df['Game_ID'] = df.apply(lambda row: f'{row.Date}-{row.Home_Team}-{row.Away_Team}', axis=1)

# ## Filter out games that have not been played yet
# df = df[df['Home_Score'] != '']

# # Replace Nan values in metrics column with empty string
# df['Metrics_Link'] = df['Metrics_Link'].fillna('')

# ### Check the length of the resulting dataframe
# # Print length of dataframe before and after filtering for exhibition games
# print(f'Length of dataframe before filtering for exhibition games: {len(df)}')

# # Filter out exhibition games - Conference = Exhibition
# df = df[df['Conference'] != 'Exhibition']

# print(f'Length of dataframe after filtering for exhibition games: {len(df)}')

In [8]:
def parse_current_season(url):
    import requests
    from bs4 import BeautifulSoup
    import pandas as pd

    # Initialize variables
    current_date = None
    current_conference = None
    game_notes = None

    # Initialize an empty list to hold the data
    data = []

    # Parse the page with BeautifulSoup
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')

    rows = soup.find_all('tr')

    # Loop through each row to find relevant information
    for row in rows:

        # Check for date row
        if row.get('class') == ['stats-section']:
            current_date = row.find('td').text.strip()
            continue

        # Check for conference row
        elif row.get('class') == ['sked-header']:
            current_conference = row.find('td').text.strip()
            continue

        # Check for game notes
        elif len(row.find_all('td')) == 2:
            game_notes = row.find_all('td')[1].text.strip()
            continue

        # Process rows with game data
        elif row.get('valign') == 'top':
            cells = row.find_all('td')
            if len(cells) >= 9:

                home_team = cells[0].text.strip().replace('-', ' ')
                home_team_link = cells[0].find('a')['href'] if cells[0].find('a') else None

                home_score = cells[1].text.strip()

                away_team = cells[3].text.strip().replace('-', ' ')
                away_team_link = cells[3].find('a')['href'] if cells[3].find('a') else None

                away_score = cells[4].text.strip()

                ot = cells[5].text.strip()

                box_link = cells[7].find('a')['href'] if cells[7].find('a') else None
                metrics_link = cells[8].find('a')['href'] if cells[8].find('a') else None

                # Capture Game Notes
                game_notes_cell = cells[-1].find('small')
                game_notes_real = game_notes_cell.text.strip() if game_notes_cell else game_notes

                # Append data to the list
                data.append([
                    current_date,
                    current_conference,
                    game_notes_real,
                    home_team,
                    home_team_link,
                    home_score,
                    away_team,
                    away_team_link,
                    away_score,
                    ot,
                    box_link,
                    metrics_link
                ])

                # Reset game notes
                game_notes = None

    # ---------------------------
    # Build the DataFrame here
    # ---------------------------

    columns = [
        'Date', 'Conference', 'Game_Notes',
        'Home_Team', 'Home_Team_Link', 'Home_Score',
        'Away_Team', 'Away_Team_Link', 'Away_Score',
        'OT', 'Box_Link', 'Metrics_Link'
    ]

    df = pd.DataFrame(data, columns=columns)

    # Extract Day of week
    df['Day'] = pd.to_datetime(df['Date']).dt.day_name()

    # Format date as YYYY-MM-DD
    df['Date'] = pd.to_datetime(df['Date']).dt.strftime('%Y-%m-%d')

    # Build Game_ID exactly as in your original code
    df['Game_ID'] = df.apply(
        lambda row: f"{row.Date}-{row.Home_Team}-{row.Away_Team}",
        axis=1
    ).str.replace(',', '')

    # Filter out games with no score
    df = df[df['Home_Score'] != '']

    # Fill NaN in metrics links
    df['Metrics_Link'] = df['Metrics_Link'].fillna('')

    # Remove exhibition
    df = df[df['Conference'] != 'Exhibition']

    return df

# ## Call to Test the function
# schedule_df = parse_current_season(current_year_url)
# schedule_df ## --- IGNORE ---


### NEW - Parse New Shot Charts

In [9]:
from typing import Optional


# Map the site's raw data-type values into a simpler categorical outcome
SHOT_TYPE_MAP = {
    "goal": "GOAL",
    "shot": "SHOT_ON_GOAL",  # on net, but not goal
    "blocked": "BLOCKED",
    "wide": "MISSED",
    "pipe": "MISSED",
    "miss": "MISSED",
}

def _parse_xy_from_style(style_str: str):
    """
    Parse --x and --y CSS variables from a style attribute.
    e.g. ' --x: 85; --y: 36; ' -> (85.0, 36.0)
    """
    if not style_str:
        return None, None

    matches = dict(re.findall(r"--(x|y)\s*:\s*([0-9.]+)", style_str))
    x = float(matches.get("x")) if "x" in matches else None
    y = float(matches.get("y")) if "y" in matches else None
    return x, y


def extract_shot_chart_from_html(
    html: str,
    game_id: str,
    home_team: Optional[str] = None,
    away_team: Optional[str] = None,
) -> pd.DataFrame:
    """
    Extract shot chart data from a box score HTML page containing the <div id="shotxy"> block.

    Returns a DataFrame with columns:
        game_id, team_side, team,
        x_pct, y_pct,
        shot_type_raw, shot_outcome,
        period, oep,
        home_starts_right
    """
    soup = BeautifulSoup(html, "lxml")
    shot_root = soup.find("div", id="shotxy")
    if shot_root is None:
        # No shot chart on this page
        return pd.DataFrame()

    container = shot_root.find("div", class_="shotxy_container")
    if container is None:
        return pd.DataFrame()

    container_classes = container.get("class", [])
    home_starts_right = "home_starts_right" in container_classes

    rows = []

    for team_div in container.select("div.shotxy_team"):
        # home / away
        team_classes = team_div.get("class", [])
        if "home" in team_classes:
            side = "home"
        elif "away" in team_classes:
            side = "away"
        else:
            side = "unknown"

        # Team name from header if you don't pass them in
        h3 = team_div.find("h3")
        label = h3.get_text(strip=True) if h3 else ""
        # "Clarkson Shooting" -> "Clarkson"
        label = re.sub(r"\s+Shooting$", "", label)

        if side == "home" and home_team:
            team_name = home_team
        elif side == "away" and away_team:
            team_name = away_team
        else:
            team_name = label

        for span in team_div.find_all("span"):
            style_str = span.get("style", "")
            x_pct, y_pct = _parse_xy_from_style(style_str)

            shot_type_raw = span.get("data-type", "").strip().lower()
            period_str = span.get("data-period", "0")
            oep = span.get("data-oep")

            try:
                period = int(period_str)
            except ValueError:
                period = None

            shot_outcome = SHOT_TYPE_MAP.get(shot_type_raw, "UNKNOWN")

            rows.append(
                {
                    "game_id": game_id,
                    "team_side": side,
                    "team": team_name,
                    "x_pct": x_pct,
                    "y_pct": y_pct,
                    "shot_type_raw": shot_type_raw,
                    "shot_outcome": shot_outcome,
                    "period": period,
                    "oep": oep,
                    "home_starts_right": home_starts_right,
                }
            )

    return pd.DataFrame(rows)


### Parsing Functions - All + Remove Dups Helper

In [10]:
## Functions for parsing the box score and metrics pages

# # Initialize logging for Error and Warning messages
# logging.basicConfig(filename='../TEMP/current_scrape.log', level=logging.INFO)

#### PARSE PLAYER STATS TABLE ####
def parse_player_summary(html_content):
    # Initialize BeautifulSoup
    soup = BeautifulSoup(html_content, 'html.parser')
    
    # Find the playersums div
    playersums_div = soup.find('div', id='playersums')
    if playersums_div is None:
        return "Player summaries div not found"

    # Initialize list to store player stats
    player_stats = []

    # Loop through each playersum div
    for player_sum in playersums_div.find_all('div', class_='playersum'):
        team = player_sum.find('td').text.strip()
        
        # Loop through table rows
        for row in player_sum.find_all('tr'):
            cols = row.find_all('td')
            if len(cols) > 1:
                player = cols[0].text.strip()
                goals = cols[1].text.strip()
                assists = cols[2].text.strip()
                points = cols[3].text.strip()
                plus_minus = cols[4].text.strip()
                shots = cols[5].text.strip()
                toi = cols[6].text.strip()
                pim = cols[7].text.strip()
                fowl = cols[8].text.strip() if len(cols) > 7 else None
                
                fow, fol = None, None
                win_percentage = None
                
                

                try:
                    if fowl and '‑' in fowl:  # Checking if it contains a hyphen
                        fow, fol = map(int, fowl.split('‑'))
                        total_fo = fow + fol
                        win_percentage = (fow / total_fo) * 100 if total_fo > 0 else 0
                except ValueError:
                    fow, fol, win_percentage = None, None, None

                

                
                player_stat = {
                    'Team': team,
                    'Player': player,
                    'G': goals,
                    'A': assists,
                    'Pt.': points,
                    '+/-': plus_minus,
                    'Sh': shots,
                    'TOI': toi,
                    'PIM': pim,
                    'FOW': fow,
                    'FOL': fol,
                    'FO%': win_percentage
                }
                player_stats.append(player_stat)

    return pd.DataFrame(player_stats)


############# PARSEING SCORING SUMMARY WITH BS4
def parse_scoring_summary(html_content):
    # Initialize BeautifulSoup
    soup = BeautifulSoup(html_content, 'html.parser')

    # Find the scoring div and table
    scoring_div = soup.find('div', id='scoring')
    if scoring_div is None:
        logging.error("Scoring div not found")
        return None

    scoring_table = scoring_div.find('table')
    if scoring_table is None:
        logging.error("Scoring table not found within the scoring div")
        return None

    # Initialize list to store scoring events
    scoring_events = []
    period = None

    # Loop through table rows
    for row in scoring_table.find_all('tr'):
        if 'stats-section' in row.get('class', []):
            td = row.find('td')
            if td:
                period = td.text.strip()
            else:
                logging.warning("Period name not found in 'stats-section' row")
                period = "Unknown"
        else:
            cols = row.find_all('td')
            if len(cols) > 1:
                try:
                    team = cols[0].text.strip()
                    pp = cols[1].text.strip()

                    player_data = cols[3].text.strip()
                    match = re.match(r"(.+)\s\((\d+)\)", player_data)
                    player = match.group(1) if match else player_data
                    goals = int(match.group(2)) if match else None

                    assist_data_raw = cols[4].text.strip()
                    assist_data = assist_data_raw.split(", ") if assist_data_raw else []
                    assist1 = assist_data[0] if len(assist_data) > 0 else None
                    assist2 = assist_data[1] if len(assist_data) > 1 else None

                    time = cols[5].text.strip()

                    scoring_event = {
                        'Period': period,
                        'Team': team,
                        'PP': pp,
                        'Player': player,
                        'Player_Goals': goals,
                        'Assist1': assist1,
                        'Assist2': assist2,
                        'Time': time
                    }
                    scoring_events.append(scoring_event)
                except Exception as e:
                    logging.error(f"An error occurred while parsing a scoring event row: {e}")
            else:
                logging.warning(f"Insufficient columns in scoring row: {len(cols)}")

    return pd.DataFrame(scoring_events)


############# PARSEING PENALTY SUMMARY WITH BS4
def parse_penalty_summary(html_content):
    # Initialize BeautifulSoup
    soup = BeautifulSoup(html_content, 'html.parser')

    # Find the penalties div and table
    penalties_div = soup.find('div', id='penalties')
    if penalties_div is None:
        logging.error("Penalties div not found")
        return None

    penalties_table = penalties_div.find('table')
    if penalties_table is None:
        logging.error("Penalties table not found within the penalties div")
        return None

    # Initialize list to store penalty events
    penalty_events = []
    period = None

    # Loop through table rows
    for row in penalties_table.find_all('tr'):
        if 'stats-section' in row.get('class', []):
            td = row.find('td')
            if td:
                period = td.text.strip()
            else:
                logging.warning("Period name not found in 'stats-section' row")
                period = "Unknown"
        else:
            cols = row.find_all('td')
            if len(cols) > 1:
                team = cols[0].text.strip()
                player = cols[1].text.strip()
                pen_length = cols[2].text.strip()
                penalty_type = cols[3].text.strip()
                time = cols[4].text.strip()

                penalty_event = {
                    'Period': period,
                    'Team': team,
                    'Player': player,
                    'Pen_Length': pen_length,
                    'Penalty_Type': penalty_type,
                    'Time': time
                }
                penalty_events.append(penalty_event)

    return pd.DataFrame(penalty_events)


############# GOALIE SUMMARY WITH BS4
def parse_goalie_stats(html_content):
    # Initialize BeautifulSoup
    soup = BeautifulSoup(html_content, 'html.parser')

    # Find the goalies div and table
    goalies_div = soup.find('div', id='goalies')
    if goalies_div is None:
        logging.error("Goalies div not found")
        return None

    goalies_table = goalies_div.find('table')
    if goalies_table is None:
        logging.error("Goalies table not found within the goalies div")
        return None

    # Initialize list to store goalie stats
    goalie_stats = []
    team = None

    # Loop through table rows
    for row in goalies_table.find_all('tr'):
        if 'stats-header' in row.get('class', []):
            td = row.find('td')
            if td:
                team = td.text.strip()
            else:
                logging.warning("Team name not found in 'stats-header' row")
                team = "Unknown"
        else:
            cols = row.find_all('td')
            if len(cols) > 1:
                goalie = cols[0].text.strip()
                sv = cols[1].text.strip()
                ga = cols[2].text.strip()
                minutes = cols[3].text.strip()

                goalie_stat = {
                    'Team': team,
                    'Goalie': goalie,
                    'SV': sv,
                    'GA': ga,
                    'Minutes': minutes
                }
                goalie_stats.append(goalie_stat)

    return pd.DataFrame(goalie_stats)


#### PARSE THE ADVANCED TEAM METRICS TABLES ####
### RETURNS WHOLE ADVANCED METRICS AS SINGLE TABLE
####################################
def parse_new_advanced_metrics(html_content):
    # Parse HTML content
    soup = BeautifulSoup(html_content, 'html.parser')
    
    # Find all tables with advanced metrics
    tables = soup.find_all('table', {'class': 'sortable metrics'})
    
    # List to store all parsed data
    all_data = []
    
    for table in tables:
        # Extract team name from the table header
        team_name = table.find('td').text.strip()
        
        # Extract headers (skipping the Player header)
        headers = [header.text for header in table.find_all('th')][1:]
        
        # Prepare final column headers
        col_names = ['Team', 'Player']
        for header in headers:
            col_names.append(header)
        
        # Extract player data
        rows = table.find_all('tr')[2:]  # skipping the two header rows
        for row in rows:
            player_data = [team_name]  # start with team name
            cells = row.find_all('td')
            player_data.append(cells[0].text.strip())  # player name
            for cell in cells[1:]:
                player_data.append(cell.text.strip())
            all_data.append(player_data)
    
    # Convert the list of data to a DataFrame
    df = pd.DataFrame(all_data, columns=col_names)
    return df

######## NEW TEST ###############  
def parse_advanced_metrics_tables(html_content):
    # Parse HTML content
    soup = BeautifulSoup(html_content, 'html.parser')
    
    # Find all tables with advanced metrics
    tables = soup.find_all('table', {'class': 'sortable metrics'})
    
    # List to store all parsed data
    all_data = []
    
    for table in tables:
        # Extract team name from the table header
        team_name = table.find('td').text.strip()
        
        # Extract headers (skipping the Player header)
        headers = [header.text for header in table.find_all('th')][1:]
        
        # Prepare final column headers
        col_names = ['Team', 'Player']
        for header in headers:
            col_names.append(header)
        
        # Extract player data
        rows = table.find_all('tr')[2:]  # skipping the two header rows
        for row in rows:
            player_data = [team_name]  # start with team name
            cells = row.find_all('td')
            player_data.append(cells[0].text.strip())  # player name
            for cell in cells[1:]:
                player_data.append(cell.text.strip())
            all_data.append(player_data)
    
    # Convert the list of data to a DataFrame
    df = pd.DataFrame(all_data, columns=col_names)
    return df

# Parsing the line chart information with specific positions for forwards and defensemen.
def parse_line_chart(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    line_chart_div = soup.find('div', id='linechart')

    if line_chart_div is None:
        logging.error("Line chart div not found")
        return pd.DataFrame()

    line_data = []

    for team_div in line_chart_div.find_all('div', recursive=False):
        h3 = team_div.find('h3')
        if h3 is None:
            logging.warning("Team name not found")
            continue
        
        team_name = h3.text.strip()
        
        for line_type_div in team_div.find_all('div', recursive=False):
            line_type = line_type_div.get('class')[0] if line_type_div.get('class') else None
            if line_type is None:
                logging.warning("Line type not found")
                continue
            
            if line_type == 'f':
                position_types = ['Left Wing', 'Center', 'Right Wing']
            elif line_type == 'd':
                position_types = ['Left D', 'Right D']
            elif line_type == 'x':
                position_types = ['Extra']
            elif line_type == 'g':
                position_types = ['Goalie']
                goalie_count = 1  # Initialize goalie count
            else:
                continue

            players = line_type_div.find_all('div')
            if not players:
                logging.warning(f"No players found for {team_name} in {line_type}")
                continue
            
            for i, player in enumerate(players):
                player_name = player.text.strip()
                if line_type == 'x':
                    player_name = player_name.split(' ')[0]
                if line_type == 'g':
                    line_number = f"Goalie {goalie_count}"
                    goalie_count += 1
                else:
                    line_number = i // len(position_types) + 1

                position = position_types[i % len(position_types)]
                line_data.append({
                    'Team': team_name,
                    'Line': line_number,
                    'Position': position,
                    'Player': player_name
                })

    if not line_data:
        logging.error("No line data was collected")

    df = pd.DataFrame(line_data)
    
    # # Log DataFrame info for debugging
    # if df.empty:
    #     logging.warning("Generated line chart DataFrame is empty.")
    # else:
    #     logging.info(f"Generated line chart DataFrame with columns: {df.columns.tolist()}")

    return df

### Get the Linescore Elements - Score, shots, ect by period####
### NEEDS UPDATE NOW THAT POSTSEASON MEAND 5th, 6th, ect PERIODS
def parse_linescore(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    linescore_data = []
    
    # Parsing the Goals table
    goals_table = soup.select_one("#goals table")
    if goals_table is None:
        logging.error("Goals table not found")
        return None
    
    rows = goals_table.select('tbody tr')
    if not rows:
        logging.warning("No rows found in Goals table")
        return None
    
    for row in rows:
        team_data = {}
        td = row.select_one('td')
        if td:
            team_data['Team'] = td.text
        else:
            logging.warning("Team name not found in Goals table")
            continue

        goals = row.select('td')[1:]
        for i, goal in enumerate(goals):
            period = i + 1
            column_name = f'goals{period}' if i < len(goals) - 1 else 'goalsT'
            team_data[column_name] = int(goal.text)
        
        linescore_data.append(team_data)
    

    # Parsing the Shots table
    shots_table = soup.select_one("#shots table")
    if shots_table is None:
        logging.error("Shots table not found")
        return None

    rows = shots_table.select('tbody tr')
    if not rows:
        logging.warning("No rows found in Shots table")
        return None

    for i, row in enumerate(rows):
        shots = row.select('td')[1:]
        if not shots:
            logging.warning(f"No shot data found for row {i+1} in Shots table")
            continue

        for j, shot in enumerate(shots):
            period = j + 1
            column_name = f'shots{period}' if j < len(shots) - 1 else 'shotsT'
            try:
                linescore_data[i][column_name] = int(shot.text.strip())
            except ValueError:
                logging.warning(f"Could not convert shot data to integer for row {i+1}, column {j+1}")
                linescore_data[i][column_name] = None

    # Parsing the PP table
    pp_table = soup.select_one("#pp table")
    if pp_table is None:
        logging.error("PP table not found")
        return None

    rows = pp_table.select('tbody tr')
    if not rows:
        logging.warning("No rows found in PP table")
        return None

    for i, row in enumerate(rows):
        try:
            pen_pim = row.select('td')[1].text.split('‑')
            linescore_data[i]['Pen'] = int(pen_pim[0])
            linescore_data[i]['PIM'] = int(pen_pim[1])

            ppg_ppo = row.select('td')[2].text.split('‑')
            linescore_data[i]['PPG'] = int(ppg_ppo[0])
            linescore_data[i]['PPO'] = int(ppg_ppo[1])

            fow_fol = row.select('td')[3].text.split('‑')
            linescore_data[i]['FOW'] = int(fow_fol[0])
            linescore_data[i]['FOL'] = int(fow_fol[1])
            linescore_data[i]['FOW%'] = (linescore_data[i]['FOW'] / (linescore_data[i]['FOW'] + linescore_data[i]['FOL'])) * 100

            ### NEW ADDITION 11-19-25 - Expected Goals have been added to this table this season - get them and store as a float
            xg_xga = row.select('td')[4].text
            linescore_data[i]['xG'] = float(xg_xga)
            

        except (ValueError, IndexError) as e:
            logging.warning(f"Could not process PP data for row {i+1}. Error: {e}")
            continue

    # Convert to DataFrame early
    df = pd.DataFrame(linescore_data)

    # Ensure all columns exist
    expected_goals_columns = [f'goals{i}' for i in range(1, 7)] + ['goalsT']
    expected_shots_columns = [f'shots{i}' for i in range(1, 7)] + ['shotsT']

    for col in expected_goals_columns + expected_shots_columns:
        if col not in df.columns:
            df[col] = 0

    # Return the final DataFrame
    return df



# Function to parse game details table
def parse_game_details(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    meta_div = soup.find('div', {'id': 'meta'})
    if meta_div is None:
        logging.error("Meta div not found")
        return None
    
    game_details_div = meta_div.find_all('div')[-1]
    if game_details_div is None:
        logging.error("Game details div not found")
        return None
    
    try:
        date_str = game_details_div.h4.string
        day_of_week, date = date_str.split(", ", 1)
        
        p_elements = game_details_div.find_all('p')
        
        # Extract conference and location details
        for p in p_elements:
            if "Game" in p.text:  # e.g., "Big Ten Game"
                details_strs = p.get_text(separator='|').split('|')
                conference = details_strs[0]
                location = details_strs[-1].split('at ')[-1]
                break
        else:  # Defaults if not found
            conference, location = None, None
        
        # Extract referees and assistant referees details
        for p in p_elements:
            if "Referees" in p.text:
                refs_str = p.strong.next_sibling if p.strong else None
                asst_refs_str = p.find_all('strong')[1].next_sibling if len(p.find_all('strong')) > 1 else None
                break
        else:  # Defaults if not found
            refs_str, asst_refs_str = None, None
        
        refs = refs_str.split(', ') if refs_str else []
        asst_refs = asst_refs_str.split(', ') if asst_refs_str else []
        refs = [re.sub(r'[^a-zA-Z ]+', '', ref).strip() for ref in refs]
        asst_refs = [re.sub(r'[^a-zA-Z ]+', '', ref).strip() for ref in asst_refs]
        
        # Extract attendance details using regex for better accuracy
        attendance_pattern = r"Attendance:\s?(\d+[\d,]*)"
        attendance_match = re.search(attendance_pattern, html_content)
        attendance = int(attendance_match.group(1).replace(',', '')) if attendance_match else None
        
        # Extract game details (like shootout results)
        details = None
        for p in p_elements:
            if "shootout" in p.text:
                details = p.text
                break
        
        # Clean details if present
        if details and '\n' in details:
            details = details.replace('\n', '').strip()
        if details and '\t' in details:
            details = re.sub('\t', ' ', details)
        
        game_details = {
            'Day': day_of_week,
            'Date': date,
            'Conference': conference,
            'Details': details,
            'Location': location,
            'Ref1': refs[0] if refs else None,
            'Ref2': refs[1] if len(refs) > 1 else None,
            'Asst_Ref1': asst_refs[0] if asst_refs else None,
            'Asst_Ref2': asst_refs[1] if len(asst_refs) > 1 else None,
            'Attendance': attendance
        }
        
        game_details_df = pd.DataFrame([game_details])
        return game_details_df

    except (AttributeError, IndexError, ValueError) as e:
        logging.error(f"Error while parsing game details: {e}")
        return None


# Parse the box score page - player stats table (G, A, Pt, +/-, Sh, PIM)
def parse_box_score(box_score_html):
    # Initialize DataFrames to None
    scoring_summary = penalty_summary = goalie_stats = player_stats = line_chart = linescore = game_details = None
    
    try:
        scoring_summary = parse_scoring_summary(box_score_html)
    except Exception as e:
        print(f"Error in parse_scoring_summary: {e}")
    
    try:
        penalty_summary = parse_penalty_summary(box_score_html)
    except Exception as e:
        print(f"Error in parse_penalty_summary: {e}")
    
    try:
        goalie_stats = parse_goalie_stats(box_score_html)
    except Exception as e:
        print(f"Error in parse_goalie_stats: {e}")
    
    try:
        player_stats = parse_player_summary(box_score_html)
    except Exception as e:
        print(f"Error in parse_player_summary: {e}")
    
    try:
        line_chart = parse_line_chart(box_score_html)
        if line_chart.empty:
            logging.info("Line chart is empty. Skipping the insert for this game.")
        else:
            logging.info(f"Line chart DataFrame structure: {line_chart.dtypes}")

        # Insert into database (make sure this part works as expected)

    except Exception as e:
        logging.error(f"Error in parse_line_chart: {e}")


    
    try:
        linescore = parse_linescore(box_score_html)
    except Exception as e:
        print(f"Error in parse_linescore: {e}")
    
    try:
        game_details = parse_game_details(box_score_html)
    except Exception as e:
        print(f"Error in parse_game_details: {e}")

    # # Add Game_ID to every DataFrame in the list
    # dataframes = [game_details, scoring_summary, penalty_summary, goalie_stats, player_stats, line_chart, linescore]
    # for df in dataframes:
    #     if df is not None and not df.empty:
    #         df["Game_ID"] = game_id

    # Combine DataFrames into a list
    all_dfs = [game_details, scoring_summary, penalty_summary, goalie_stats, player_stats, line_chart, linescore]
    
    return all_dfs


# helper function to rename duplicate columns
def rename_duplicate_columns(df):
    cols = pd.Series(df.columns)
    for dup in df.columns[df.columns.duplicated()].unique(): 
        cols[df.columns.get_loc(dup)] = [f"{dup}_{i}" if i != 0 else dup for i in range(df.columns.get_loc(dup).sum())]
    df.columns = cols
    return df

### Clean Advanced Metrics Table - helper

In [11]:
def clean_advanced_metrics(df: pd.DataFrame, game_id: str) -> pd.DataFrame:
    """
    Apply all the transformation logic from the original notebook to make
    advanced metrics compatible with the existing DB schema.
    """

    # --------------------------------------------------------
    # 1. Attach Game_ID
    # --------------------------------------------------------
    df["Game_ID"] = game_id

    # --------------------------------------------------------
    # 2. Replace team abbreviations with full names
    # (you already have this mapping defined in original notebook)
    # --------------------------------------------------------
    df = replace_abbreviations_with_fullnames(df, "Team", abbreviation_to_fullname)

    # --------------------------------------------------------
    # 3. Rename columns into the canonical schema
    # Your original notebook listed these EXACT names:
    # --------------------------------------------------------
    new_names = [
        'Team', 'Player',
        'TOTAL_Block', 'TOTAL_Miss', 'TOTAL_Saved', 'TOTAL_Goals', 'TOTAL_Total_Shots',
        'EVEN_Block', 'EVEN_Miss', 'EVEN_Saved', 'EVEN_Goals', 'EVEN_Total_Shots',
        'PP_Block', 'PP_Miss', 'PP_Saved', 'PP_Goals', 'PP_Total_Shots',
        'CLOSE_Block', 'CLOSE_Miss', 'CLOSE_Saved', 'CLOSE_Goals', 'CLOSE_Total_Shots',
        'D_Blocks', 'Faceoffs', 'Game_ID'
    ]

    if len(df.columns) != len(new_names):
        logging.error(
            f"Advanced metrics columns mismatch. "
            f"Expected {len(new_names)} columns but got {len(df.columns)}."
        )
        return None

    df.columns = new_names

    # --------------------------------------------------------
    # 4. Fill NaN with 0
    # --------------------------------------------------------
    df = df.fillna(0)

    return df


### DB Helpers

In [12]:
# ==========================================
# 2. DB helpers
# ==========================================
from sqlalchemy import inspect

def get_engine(db_path: str):
    """Create a SQLAlchemy engine for the given SQLite DB path."""
    return create_engine(f"sqlite:///{db_path}")


def get_existing_game_ids(db_path: str, table: str = CANONICAL_GAME_TABLE) -> set:
    """
    Return a set of Game_ID strings that already exist in the canonical table.
    If the table does not exist yet, return an empty set.
    """
    engine = get_engine(db_path)
    insp = inspect(engine)

    if not insp.has_table(table):
        logging.info(f"Table '{table}' does not exist yet; treating as empty DB.")
        return set()

    with engine.connect() as conn:
        df_ids = pd.read_sql(f"SELECT DISTINCT Game_ID FROM {table}", conn)

    # Ensure string type for consistent comparisons
    return set(df_ids["Game_ID"].astype(str))


def delete_game_from_db(db_path: str, game_id: str, tables):
    """
    Delete rows for a given Game_ID from the specified tables.
    Skips tables that do not yet exist (e.g., newly-added tables).
    """
    engine = get_engine(db_path)

    with engine.begin() as conn:
        for table in tables:
            try:
                # Check if table exists in this SQLite DB
                exists = conn.execute(
                    text(
                        "SELECT name FROM sqlite_master "
                        "WHERE type='table' AND name = :tname"
                    ),
                    {"tname": table},
                ).scalar()

                if not exists:
                    # New table that hasn't been created yet – just skip
                    logging.info(
                        f"Table '{table}' does not exist yet; "
                        f"skipping delete for Game_ID={game_id}."
                    )
                    continue

                conn.execute(
                    text(f"DELETE FROM {table} WHERE Game_ID = :gid"),
                    {"gid": game_id},
                )
                logging.info(f"Deleted rows for Game_ID={game_id} from {table}.")

            except OperationalError as e:
                # Extra safety net; log and continue
                logging.warning(
                    f"OperationalError deleting from {table} for Game_ID={game_id}: {e}"
                )
                continue

# def delete_game_from_db(db_path: str, game_id: str, tables: list[str]):
#     """
#     Delete all rows for a given Game_ID from the specified tables.
#     Used when rescraping a game within the update window.
#     """
#     engine = get_engine(db_path)
#     with engine.begin() as conn:  # transaction
#         for table in tables:
#             logging.info(f"Deleting Game_ID={game_id} from {table}")
#             conn.execute(
#                 text(f"DELETE FROM {table} WHERE Game_ID = :gid"),
#                 {"gid": game_id}
#             )


def append_game_dfs_to_db(db_path: str, df_list, table_names, game_id: str):
    engine = get_engine(db_path)

    for df, table in zip(df_list, table_names):
        if df is None or len(df) == 0:
            logging.warning(f"Skipping empty df for table={table}")
            continue
        
        # Make sure every DataFrame has Game_ID column with a value
    # --- 0. Ensure canonical Game_ID and drop any variants (game_id, Game_id, etc.) ---
        gid_like_cols = [c for c in df.columns if c.lower() == "game_id"]
        for c in gid_like_cols:
            df = df.drop(columns=[c])

        # Now set the canonical Game_ID column for all tables
        df["Game_ID"] = game_id

        
        # --- 1. Normalize column names (Pt. -> Pts, FO% -> FO_Pct, etc.) ---
        df = normalize_columns(df, table)

        # ------------------------------------------------------
        # SPECIAL HANDLING: game_details → add Home & Away cols
        # ------------------------------------------------------
        if table == "game_details":
            if "Game_ID" in df.columns:
                df["Away_Team"] = df["Game_ID"].apply(lambda x: str(x).split("-")[3] if isinstance(x, str) and len(str(x).split("-")) >= 5 else None)
                df["Home_Team"] = df["Game_ID"].apply(lambda x: str(x).split("-")[4] if isinstance(x, str) and len(str(x).split("-")) >= 5 else None)

        # ------------------------------------------------------
        # SPECIAL HANDLING: expand team abbreviations
        # ------------------------------------------------------
        if table in ("linescore", "penalty_summary", "scoring_summary", "shot_events"):
            if "Team" in df.columns:
                df = replace_abbreviations_with_fullnames(df, "Team", abbreviation_to_fullname)

        # ------------------------------------------------------
        # SPECIAL HANDLING: scoring_summary → add Home & Away
        # ------------------------------------------------------
        if table in ("scoring_summary", "game_details", "advanced_metrics", "line_chart"):
            if "Game_ID" in df.columns:
                df["Away_Team"] = df["Game_ID"].apply(lambda x: str(x).split("-")[3] if isinstance(x, str) and len(str(x).split("-")) >= 5 else None)
                df["Home_Team"] = df["Game_ID"].apply(lambda x: str(x).split("-")[4] if isinstance(x, str) and len(str(x).split("-")) >= 5 else None)



        # --- 2. Remove header/noise rows in player_stats ---
        df = drop_header_and_noise_rows(df, table)

        # --- 3. Deduplicate column names (e.g. 'Shots', 'Shots_1') ---
        df = rename_duplicate_columns(df)

        # --- 4. Add missing columns to DB (migration-safe) ---
        ensure_table_columns(engine, table, list(map(str, df.columns)))

        # NEW: clean troublesome characters in team columns
        cols_to_clean = TEAM_CLEAN_COLUMNS.get(table, [])
        if cols_to_clean:
            df = clean_name_columns(df, cols_to_clean)

    

        logging.info(f"Appending {len(df)} rows to table={table}")
        df.to_sql(table, engine, if_exists="append", index=False)


## Filter games that need scraping

In [13]:
# # ==========================================
# # 3. Determine which games to scrape
# # ==========================================

### New Code to ignore games with no Metrics_Link
def classify_games_to_scrape(
    schedule_df: pd.DataFrame,
    existing_ids: set,
    rescrape_window_days: int | None = None,
) -> pd.DataFrame:
    """
    Add a 'scrape_action' column to schedule_df with values:
      - 'new'    : not in existing_ids
      - 'update' : in existing_ids and within rescrape_window_days
      - 'skip'   : in existing_ids and older than the window,
                   OR if Metrics_Link is missing/empty.
    """

    df = schedule_df.copy()

    # Make sure Date is a datetime
    df["Game_Date"] = pd.to_datetime(df["Date"], errors="coerce")

    # Normalize the Metrics_Link column and detect missing URLs
    df["Metrics_Link"] = df["Metrics_Link"].astype(str).str.strip()
    df["has_metrics_link"] = df["Metrics_Link"].notna() & df["Metrics_Link"].ne("")

    today = pd.Timestamp.today().normalize()
    cutoff = today - pd.Timedelta(days=rescrape_window_days) if rescrape_window_days else None

    def decide(row):
        gid = str(row["Game_ID"])

        # 🚫 NEW RULE: if there is NO Metrics_Link, never try to scrape it
        if not row["has_metrics_link"]:
            return "skip"

        # Standard logic
        if gid not in existing_ids:
            return "new"
        if cutoff is not None and pd.notna(row["Game_Date"]) and row["Game_Date"] >= cutoff:
            return "update"
        return "skip"

    df["scrape_action"] = df.apply(decide, axis=1)
    return df

# def classify_games_to_scrape(
#     schedule_df: pd.DataFrame,
#     existing_ids: set,
#     rescrape_window_days: int | None = None,
# ) -> pd.DataFrame:
#     """
#     Add a 'scrape_action' column to schedule_df with values:
#       - 'new'    : not in existing_ids
#       - 'update' : in existing_ids and within rescrape_window_days
#       - 'skip'   : in existing_ids and older than the window
#     """

#     df = schedule_df.copy()

#     # Make sure Date is a datetime
#     df["Game_Date"] = pd.to_datetime(df["Date"], errors="coerce")

#     today = pd.Timestamp.today().normalize()
#     cutoff = None
#     if rescrape_window_days is not None:
#         cutoff = today - pd.Timedelta(days=rescrape_window_days)

#     def decide(row):
#         gid = str(row["Game_ID"])
#         if gid not in existing_ids:
#             return "new"
#         if cutoff is not None and pd.notna(row["Game_Date"]) and row["Game_Date"] >= cutoff:
#             return "update"
#         return "skip"

#     df["scrape_action"] = df.apply(decide, axis=1)
#     return df

## Main Scraping Logic

In [14]:
def scrape_needed_games():
    # 4.1 Load existing Game_IDs from DB
    existing_ids = get_existing_game_ids(recent_clean_db, table=CANONICAL_GAME_TABLE)
    logging.info(f"Found {len(existing_ids)} existing Game_IDs in DB.")

    # 4.2 Parse the current season schedule
    schedule_df = parse_current_season(SCHEDULE_URL)

    # Optional sanity filter, if your parse_current_season doesn't already:
    schedule_df = schedule_df[schedule_df["Home_Score"] != ""]

    logging.info(f"Schedule contains {len(schedule_df)} completed games.")

    # 4.3 Classify games
    schedule_df = classify_games_to_scrape(
        schedule_df,
        existing_ids=existing_ids,
        rescrape_window_days=RESCRAPE_WINDOW_DAYS,
    )

    # What are we actually going to touch?
    to_scrape = schedule_df[schedule_df["scrape_action"].isin(["new", "update"])].copy()
    logging.info(
        f"Games to scrape this run: total={len(to_scrape)}, "
        f"new={sum(to_scrape['scrape_action']=='new')}, "
        f"update={sum(to_scrape['scrape_action']=='update')}"
    )

    if to_scrape.empty:
        logging.info("No new or recent games to scrape. Done.")
        return

    # 4.4 Iterate through the games we need, scraping & writing to DB
    for row in tqdm(to_scrape.itertuples(), total=len(to_scrape)):
        game_id = str(row.Game_ID)
        action = row.scrape_action

        # --- BOX SCORE URL (primary for all base tables) ---
        box_rel_link = getattr(row, "Box_Link", "")  # relative href, e.g. '/box/final/...'
        if not box_rel_link:
            logging.warning(f"No Box_Link for Game_ID={game_id}, skipping base tables.")
            continue

        box_url = BASE_URL + box_rel_link
        logging.info(f"[{action.upper()}] Scraping box score for Game_ID={game_id} from {box_url}")

        try:
            box_resp = requests.get(box_url)
            box_resp.raise_for_status()
        except Exception as e:
            logging.error(f"Failed to fetch box score URL {box_url} for Game_ID={game_id}: {e}")
            continue

        # --- Parse the box score HTML using your existing function ---
        try:
            box_score_dfs = parse_box_score(box_resp.text)
        except Exception as e:
            logging.error(f"parse_box_score failed for Game_ID={game_id}: {e}")
            continue

        if box_score_dfs is None:
            logging.error(f"parse_box_score returned None for Game_ID={game_id}; skipping.")
            continue

        # Ensure list/tuple
        if not isinstance(box_score_dfs, (list, tuple)):
            logging.error(
                f"parse_box_score returned {type(box_score_dfs)} for Game_ID={game_id}; expected list/tuple."
            )
            continue

        box_score_dfs = list(box_score_dfs)

        # Optional sanity check against GAME_TABLES length
        if len(box_score_dfs) != len(GAME_TABLES):
            logging.error(
                f"parse_box_score returned {len(box_score_dfs)} tables but expected "
                f"{len(GAME_TABLES)} for Game_ID={game_id}; skipping."
            )
            continue

        # If 'update', delete old rows first (base tables)
        if action == "update":
            delete_game_from_db(recent_clean_db, game_id, GAME_TABLES)

        # Append the base tables (game_details, scoring, penalty, goalie, players, line_chart, linescore)
        append_game_dfs_to_db(recent_clean_db, box_score_dfs, GAME_TABLES, game_id)

        # ------------------------------------------------------------------
        # NEW: Shot chart (shot_events) from the same box score HTML
        # ------------------------------------------------------------------
        # Try to grab canonical home/away names from the schedule row if present,
        # but fall back to whatever is in the shot chart h3 labels.
        home_team = getattr(row, "Home_Team", None)
        away_team = getattr(row, "Away_Team", None)

        try:
            shot_df = extract_shot_chart_from_html(
                html=box_resp.text,
                game_id=game_id,
                home_team=home_team,
                away_team=away_team,
            )
        except Exception as e:
            logging.error(f"extract_shot_chart_from_html failed for Game_ID={game_id}: {e}")
            shot_df = None

        if shot_df is not None and not shot_df.empty:
            logging.info(f"Extracted {len(shot_df)} shot events for Game_ID={game_id}.")

            # On update, clear old shot_events rows for this game
            if action == "update":
                delete_game_from_db(recent_clean_db, game_id, ["shot_events"])

            # Reuse the same append helper so Game_ID, column normalization,
            # and table migrations behave consistently.
            append_game_dfs_to_db(
                recent_clean_db,
                [shot_df],
                ["shot_events"],
                game_id,
            )
        else:
            logging.info(f"No shot events found for Game_ID={game_id} (or shot chart missing).")

        # --- OPTIONAL: Advanced metrics from `Metrics_Link` ---
        metrics_rel_link = getattr(row, "Metrics_Link", "")
        if metrics_rel_link:
            metrics_url = BASE_URL + metrics_rel_link

            try:
                metrics_resp = requests.get(metrics_url)
                metrics_resp.raise_for_status()
                raw_adv_df = parse_advanced_metrics_tables(metrics_resp.text)

                if raw_adv_df is not None and not raw_adv_df.empty:
                    adv_df = clean_advanced_metrics(raw_adv_df, game_id)

                    if adv_df is not None:
                        engine = get_engine(recent_clean_db)

                        # Use an inspector to see if the table already exists
                        inspector = inspect(engine)
                        table_exists = inspector.has_table("advanced_metrics")

                        with engine.begin() as conn:
                            # Only try to delete if the table already exists
                            if table_exists:
                                conn.execute(
                                    text(
                                        "DELETE FROM advanced_metrics WHERE Game_ID = :gid"
                                    ),
                                    {"gid": game_id},
                                )

                        # This will create the table automatically if it doesn't exist
                        adv_df.to_sql(
                            "advanced_metrics",
                            engine,
                            if_exists="append",
                            index=False,
                        )

            except Exception as e:
                logging.error(
                    f"Failed to parse advanced metrics for Game_ID={game_id}: {e}"
                )

        # # --- OPTIONAL: Advanced metrics from `Metrics_Link` ---
        # metrics_rel_link = getattr(row, "Metrics_Link", "")
        # if metrics_rel_link:
        #     metrics_url = BASE_URL + metrics_rel_link

        #     try:
        #         metrics_resp = requests.get(metrics_url)
        #         metrics_resp.raise_for_status()
        #         raw_adv_df = parse_advanced_metrics_tables(metrics_resp.text)

        #         if raw_adv_df is not None and not raw_adv_df.empty:
        #             adv_df = clean_advanced_metrics(raw_adv_df, game_id)

        #             if adv_df is not None:
        #                 engine = get_engine(recent_clean_db)

        #                 # 🔑 Delete any existing advanced_metrics rows for this game_id
        #                 with engine.begin() as conn:
        #                     conn.execute(
        #                         text("DELETE FROM advanced_metrics WHERE Game_ID = :gid"),
        #                         {"gid": game_id}
        #                     )

        #                 # Now append the fresh rows
        #                 adv_df.to_sql("advanced_metrics", engine, if_exists="append", index=False)

        #     except Exception as e:
        #         logging.error(f"Failed to parse advanced metrics for Game_ID={game_id}: {e}")

        # Be nice to CHN
        time.sleep(0.8)
        # You have this twice already, leaving it as-is:
        time.sleep(0.8)  # tune as needed

    logging.info("Incremental scrape complete.")


## Call and Run

In [15]:
# ==========================================
# 5. Entry point cell for the notebook
# ==========================================
if __name__ == "__main__":
    scrape_needed_games()

### NOTE DEC 5: fThe code fails if one of the games in the scrape list
### is does not have a Final Box and still links to the "Live Box"
### Live Box turns to Final box within a few hours after the game ends.
### Need to add error handling to skip these games and log them

100%|██████████| 53/53 [03:11<00:00,  3.60s/it]


In [16]:
#### Go through each table and replace abbreviation with full team name ###


In [17]:
def update_player_stats_ytd(db_path: str):
    """
    Recompute the player_stats_ytd summary table from the full player_stats table.
    Safe to call after each incremental scrape.
    """
    engine = get_engine(db_path)

    # 1. Load all player stats
    df = pd.read_sql("SELECT * FROM player_stats", engine)

    if df.empty:
        logging.warning("player_stats table is empty; skipping YTD aggregation.")
        return

    # 2. Clean player name (important for matching!)
    df["Clean_Player"] = df["Player"].astype(str).str.replace("\xa0", " ", regex=False).str.strip()

    # 3. Remove rows where player == team (ghost header rows)
    df = df[df["Player"] != df["Team"]]

    # 4. Convert numeric columns
    numeric_cols = ["G", "A", "Pts", "PlusMinus", "Shots", "PIM", "FOW", "FOL"]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

    # 5. Convert TOI (MM:SS) -> seconds
    if "TOI" in df.columns:
        df["TOI"] = pd.to_datetime(df["TOI"], format="%M:%S", errors="coerce").dt.time
        df["TOI_sec"] = df["TOI"].apply(
            lambda t: t.minute * 60 + t.second if pd.notnull(t) else 0
        )
    else:
        df["TOI_sec"] = 0

    # 6. Group to compute YTD aggregates
    ytd = (
        df.groupby(["Clean_Player", "Team"])
        .agg(
            G=("G", "sum"),
            A=("A", "sum"),
            Pts=("Pts", "sum"),
            PlusMinus=("PlusMinus", "sum"),
            Shots=("Shots", "sum"),
            TOI_sec=("TOI_sec", "sum"),
            PIM=("PIM", "sum"),
            FOW=("FOW", "sum"),
            FOL=("FOL", "sum"),
            Games_Played=("Game_ID", "count"),
        )
        .reset_index()
    )

    # 7. Compute FO%
    ytd["FO%"] = (ytd["FOW"] / (ytd["FOW"] + ytd["FOL"]).replace(0, np.nan)) * 100
    ytd["FO%"] = ytd["FO%"].fillna(0)

    # 8. Convert TOI back to HH:MM:SS
    ytd["TOI"] = pd.to_datetime(ytd["TOI_sec"], unit="s").dt.strftime("%H:%M:%S")

    # 9. Write back to DB
    ytd.to_sql("player_stats_ytd", engine, if_exists="replace", index=False)

    logging.info(f"Updated player_stats_ytd ({len(ytd)} rows)")


In [18]:
# Create a dictionary for abbreviations to full team names
abbreviation_to_fullname = school_info_df.set_index('abv')['School'].to_dict()

# Define a function to replace abbreviations in a column with full team names
def replace_abbreviations_with_fullnames(df, column_name, abbreviation_dict):
    """
    Replaces abbreviations in the specified column of a DataFrame with full team names.
    
    Args:
        df (pd.DataFrame): The DataFrame containing the column to process.
        column_name (str): The column name where abbreviations need to be replaced.
        abbreviation_dict (dict): Dictionary mapping abbreviations to full names.
    
    Returns:
        pd.DataFrame: DataFrame with updated column values.
    """
    df[column_name] = df[column_name].replace(abbreviation_dict)
    return df

In [19]:
### add conn to variable list
conn = get_engine(recent_clean_db).connect()

## Add The primary team names to the linescores table
# Read the linescores table into a DataFrame
df_linescores = pd.read_sql("SELECT * FROM linescore", conn)
# # Apply the dictionary to the Team column
# df_linescores['Team'] = df_linescores['Team'].apply(lambda x: matched_dict[x])
# Apply the replace abbreviation function to the Team column
df_linescores = replace_abbreviations_with_fullnames(df_linescores, 'Team', abbreviation_to_fullname)

# # Apply same method to penalty_summary:
df_penalty = pd.read_sql("SELECT * FROM penalty_summary", conn)
# df_penalty['Team'] = df_penalty['Team'].apply(lambda x: matched_dict[x]) # Apply the dictionary to the Team column
df_penalty = replace_abbreviations_with_fullnames(df_penalty, 'Team', abbreviation_to_fullname)

# # Apply same method to scorring_summary:
df_scoring = pd.read_sql("SELECT * FROM scoring_summary", conn)
# df_scoring['Team'] = df_scoring['Team'].apply(lambda x: matched_dict[x])
df_scoring = replace_abbreviations_with_fullnames(df_scoring, 'Team', abbreviation_to_fullname)


In [20]:
logging.info("Recomputing player year-to-date stats...")
update_player_stats_ytd(recent_clean_db)
